### Quality check analysis of anndata object from re-mapped E-MTAB-9543 samples
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Creation date:** 2nd of December 2024
- **Last modified date:** 2nd of December 2024

This notebook contains such parts:
* Doublets score prediction with `scrublet`
* Calculating quality metrics with `sctk`
* Identify cells that pass QC on:
    * Individual cell level
    * QC cluster level
    * Both individual cell and cluster levels
* Visualize quality metrics before filtering
* Filter out cells that do not pass QC
* Visualize quality metrics after filtering
* Identify sex covariates
* Calculate cell cycle score

### Import packages

In [121]:
import scanpy as sc
import scrublet as scr
import numpy as np
import pandas as pd
import sctk
import muon as mu
from datetime import datetime
import os
import json
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import anndata as ad

### Set up parameters to save plots and objects

In [122]:
project = 'gut'
species = 'hs'
atribute = 'Elementaite2021_E-MTAB-9543'
name = 'AM'
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
counts = 'raw'

### Load data

In [ ]:
adata = sc.read_h5ad('raw_data/Elmentaite_2021/gut_hs_Elementaite2021_E-MTAB-9543_AM_05112024_113505_raw.h5ad')
adata

AnnData object with n_obs × n_vars = 139345920 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library

## Basic filtering

In [124]:
sc.pp.filter_cells(adata, min_genes=100)

## Doublet score prediction

In [125]:
adata.obs['Source Name'].value_counts()

Source Name
Human_colon_16S8117828    32766
Human_colon_16S8001903    14619
Human_colon_16S8117831    13359
Human_colon_16S8117830    12456
Human_colon_16S8000513    11823
                          ...  
Human_colon_16S8002582      447
Human_colon_16S8002581      324
Human_colon_16S8002574       24
Human_colon_16S8002566       18
Human_colon_16S8002573        6
Name: count, Length: 63, dtype: int64

In [126]:
skip_samples = ["Human_colon_16S8002574", "Human_colon_16S8002566", "Human_colon_16S8002573"]

adata.obs['doublet_scores'] = np.nan
adata.obs['predicted_doublets'] = np.nan

sample_names = adata.obs['Source Name'].unique()

for sample_name in sample_names:
    if sample_name in skip_samples:
        print(f"Skipping sample '{sample_name}' as requested.")
        continue

    mask = adata.obs['Source Name'] == sample_name
    sample_adata = adata[mask].copy()

    scrub = scr.Scrublet(sample_adata.X)

    sample_adata.obs['doublet_scores'], sample_adata.obs['predicted_doublets'] = scrub.scrub_doublets()

    adata.obs.loc[mask, 'doublet_scores'] = sample_adata.obs['doublet_scores']
    adata.obs.loc[mask, 'predicted_doublets'] = sample_adata.obs['predicted_doublets']

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.40
Detected doublet rate = 0.4%
Estimated detectable doublet fraction = 4.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 9.2%
Elapsed time: 0.4 seconds
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.37
Detected doublet rate = 1.3%
Estimated detectable doublet fraction = 15.1%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 8.6%
Elapsed time: 0.4 seconds
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.39
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 7.0%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 2.8%
Elapsed time: 0.5 seconds
Preprocessing...
Simulating doublets

In [127]:
doub_tab = pd.crosstab(adata.obs['Source Name'],adata.obs['predicted_doublets'])
doub_tab.sum()

predicted_doublets
False    239046
True        825
dtype: int64

In [128]:
doub_tab_percentage = doub_tab.div(doub_tab.sum(axis=1), axis=0) * 100

In [129]:
doub_tab_percentage

predicted_doublets,False,True
Source Name,,
Human_colon_16S8000471,99.613900,0.386100
Human_colon_16S8000473,98.706897,1.293103
Human_colon_16S8000475,99.805447,0.194553
Human_colon_16S8000477,99.772210,0.227790
Human_colon_16S8000479,99.115044,0.884956
Human_colon_16S8000481,99.522293,0.477707
Human_colon_16S8000483,97.727273,2.272727
Human_colon_16S8000487,98.452012,1.547988
Human_colon_16S8000489,98.148148,1.851852


In [ ]:
plt.figure(figsize=(15, 7), dpi = 300)
sns.barplot(x=doub_tab_percentage.index, y=doub_tab_percentage[True])
plt.ylim(0, 100)
plt.xticks(rotation=90)
plt.xlabel('Source Name')
plt.ylabel('Percentage of Doublets')
plt.title('Percentage of Doublets per Sample')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_scrublet_doublets_{timestamp}_.png")
plt.show()

## Quality Check

+ Calculate QC metrics

In [131]:
sctk.calculate_qc(adata)

In [132]:
adata.obs['total_counts'] = adata.X.sum(axis=1)

In [133]:
adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(axis=1)

+ Determine which of the cells pass QC individually

In [134]:
sctk.cellwise_qc(adata)
adata

n_counts: [979.4897176125033, 70684.015625], 50523/239919 passed
n_genes: [99.9999893018715, 6007.00244140625], 239919/239919 passed
percent_mito: [0.0, 20.606064077845623], 101700/239919 passed
percent_ribo: [0.0, 14.892585044386555], 237450/239919 passed
percent_hb: [0.0, 0.003914146437626081], 227517/239919 passed
7929/239919 pass


AnnData object with n_obs × n_vars = 239919 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library co

In [135]:
adata.obs['cell_passed_qc'].sum()

7929

In [136]:
adata.uns['scautoqc_ranges']

,low,high
n_counts,979.489718,70684.015625
n_genes,99.999989,6007.002441
percent_mito,0.0,20.606064
percent_ribo,0.0,14.892585
percent_hb,0.0,0.003914


+ Check the QC parameters (default parameters)

In [137]:
sctk.default_metric_params_df

,min,max,scale,side,min_pass_rate
n_counts,1000.00,NaN,log,min_only,0.10
n_genes,100.00,NaN,log,min_only,0.10
percent_mito,0.01,20.0,log,max_only,0.10
percent_ribo,0.00,100.0,log,both,0.10
percent_hb,NaN,1.0,log,max_only,0.10
percent_soup,NaN,5.0,log,max_only,0.10
percent_spliced,50.00,97.5,log,both,0.10
scrublet_score,NaN,0.3,linear,max_only,0.95


### Visualize QC metrics before filtering

In [ ]:
plt.figure(figsize=(15, 15), dpi=300)
sns.scatterplot(data=adata.obs, x='total_counts', y='n_genes_by_counts' , hue ='Source Name', alpha = 0.4, s=4)
plt.legend(title='sample', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(0, int(max(adata.obs['total_counts'])) + 1, 15000),rotation=90, fontsize = 10)
plt.yticks(range(0, int(max(adata.obs['n_genes_by_counts'])) + 1, 1000),fontsize = 10)
plt.title(f'Counts vs Genes - Before filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_counts_vs_genes_before_filtering_{timestamp}.png", bbox_inches="tight")
plt.show()

In [ ]:
variables = 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'n_genes_by_counts', 'total_counts', 'percent_mito', 'percent_ribo', 'percent_hb', 'doublet_scores'

for var in variables:

    fig, ax = plt.subplots(figsize=(15, 10), ncols=2, gridspec_kw={'width_ratios': [4, 2]})

    sns.violinplot(data=adata.obs, x='Source Name', y=var, ax=ax[0])
   
    medians = adata.obs.groupby('Source Name')[var].median()
    
    ax[0].set_title(f'Violin Plot of {var} by Sample - Before filtering')
    ax[0].set_xlabel('Sample')
    ax[0].set_ylabel(var)
    ax[0].tick_params(axis='x', rotation=90)

    median_df = pd.DataFrame({'Sample': medians.index, 'Median': medians.values})

    ax[1].axis('off')
    ax[1].table(cellText=median_df.values, colLabels=median_df.columns, loc='center')
    ax[1].set_title('Median Values')
    
    plt.tight_layout()
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_{var}_before_filtering_{timestamp}.png", dpi = 300, bbox_inches="tight")
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata.obs, x='Source Name')
plt.ylim(0, 35000)
plt.xticks(rotation=90)
plt.title('Number of Cells per Sample - Before filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_number_of_cells_before_filtering_{timestamp}.png", bbox_inches="tight")

### Filter cells that do not pass QC

In [141]:
adata_filtered = adata[adata.obs['percent_mito'] < 50].copy()

In [142]:
adata_filtered = adata_filtered[adata_filtered.obs['n_genes'] > 50].copy()
adata_filtered = adata_filtered[adata_filtered.obs['n_counts'] > 500].copy()

In [143]:
adata_filtered = adata_filtered[adata_filtered.obs['percent_hb'] < 2].copy()

In [144]:
adata_filtered = adata_filtered[adata_filtered.obs['total_counts'] < 9000].copy()

In [145]:
adata_filtered = adata_filtered[adata_filtered.obs['n_genes_by_counts'] < 5000].copy()

In [146]:
adata_filtered

AnnData object with n_obs × n_vars = 62586 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library con

### Visualize QC metrics after filtering

In [ ]:
plt.figure(figsize=(15, 15))
sns.scatterplot(data=adata_filtered.obs, x='total_counts', y='n_genes_by_counts' , hue ='Source Name', alpha = 0.4, s=4)
plt.legend(title='sample', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(0, int(max(adata_filtered.obs['total_counts'])) + 1, 3000),rotation=90, fontsize = 10)
plt.yticks(range(0, int(max(adata_filtered.obs['n_genes_by_counts'])) + 1, 1000),fontsize = 10)
plt.title(f'Counts vs Genes - After filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_counts_vs_genes_after_filtering_{timestamp}.png", bbox_inches="tight")
plt.show()

In [ ]:
variables = 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'n_genes_by_counts', 'total_counts', 'percent_mito', 'percent_ribo', 'percent_hb', 'doublet_scores'

for var in variables:

    fig, ax = plt.subplots(figsize=(15, 10), ncols=2, gridspec_kw={'width_ratios': [4, 1]})

    sns.violinplot(data=adata_filtered.obs, x='Source Name', y=var, ax=ax[0])
   
    medians = adata_filtered.obs.groupby('Source Name')[var].median()
    
    ax[0].set_title(f'Violin Plot of {var} by Sample - After filtering')
    ax[0].set_xlabel('Sample')
    ax[0].set_ylabel(var)
    ax[0].tick_params(axis='x', rotation=90)

    median_df = pd.DataFrame({'Sample': medians.index, 'Median': medians.values})

    ax[1].axis('off')
    ax[1].table(cellText=median_df.values, colLabels=median_df.columns, loc='center')
    ax[1].set_title('Median Values')
    
    plt.tight_layout()
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_{var}_after_filtering_{timestamp}.png", dpi = 300, bbox_inches="tight")
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata_filtered.obs, x='Source Name')
plt.ylim(0, 12000)
plt.xticks(rotation=90)
plt.title('Number of Cells per Sample - After filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_number_of_cells_after_filtering_{timestamp}.png", bbox_inches="tight")

## Sex covariate analysis

In [150]:
annot = sc.queries.biomart_annotations(
        "hsapiens",
        ["ensembl_gene_id", "external_gene_name", "start_position", "end_position", "chromosome_name"],
    ).set_index("external_gene_name")

* Chr Y genes calculation

In [151]:
chrY_genes = adata_filtered.var_names.intersection(annot.index[annot.chromosome_name == "Y"])

In [152]:
adata_filtered.obs['percent_chrY'] = np.sum(
    adata_filtered[:, chrY_genes].X, axis = 1) / np.sum(adata_filtered.X, axis = 1) * 100

+ XIST expression analysis

In [153]:
xist_counts = adata_filtered.X[:, adata_filtered.var_names.str.match('XIST')].toarray()

In [154]:
xist_percentage = (np.sum(xist_counts, axis=1) / adata_filtered.obs['total_counts']) * 100

In [155]:
xist_counts_series = pd.Series(xist_counts.squeeze(), index = adata_filtered.obs_names, name = "XIST-counts")
xist_percentage_series = pd.Series(xist_percentage, index=adata_filtered.obs_names, name="percent_XIST")

adata_filtered.obs["XIST-counts"] = xist_counts_series
adata_filtered.obs["XIST-percentage"] = xist_percentage_series

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.violin(adata_filtered, ["XIST-counts", "percent_chrY"], jitter = 0.4, groupby = 'Source Name', rotation = 90, show=False)
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_sex_covariates_{timestamp}_.png", bbox_inches="tight")
    plt.show()

In [157]:
average_xist_counts = adata_filtered.obs.groupby('Source Name')['XIST-counts'].mean()

In [158]:
average_percent_chrY = adata_filtered.obs.groupby('Source Name')['percent_chrY'].mean()

In [159]:
scaled_xist_counts = (average_xist_counts - average_xist_counts.min()) / (average_xist_counts.max() - average_xist_counts.min())
scaled_percent_chrY = (average_percent_chrY - average_percent_chrY.min()) / (average_percent_chrY.max() - average_percent_chrY.min())

In [160]:
sample_sex = pd.DataFrame({
    'scaled_XIST_counts': scaled_xist_counts,
    'scaled_percent_chrY': scaled_percent_chrY
})
sample_sex['sex'] = np.where(sample_sex['scaled_XIST_counts'] > sample_sex['scaled_percent_chrY'], 'female', 'male')

sample_sex

,scaled_XIST_counts,scaled_percent_chrY,sex
Source Name,,,
Human_colon_16S8000471,0.000000,0.104334,male
Human_colon_16S8000473,0.000000,0.214192,male
Human_colon_16S8000475,0.000000,0.250858,male
Human_colon_16S8000477,0.000000,0.438016,male
Human_colon_16S8000479,0.000000,0.272117,male
Human_colon_16S8000481,0.000000,0.141543,male
Human_colon_16S8000483,0.000000,0.201754,male
Human_colon_16S8000487,0.000000,0.139972,male
Human_colon_16S8000489,0.000000,0.000000,male


In [161]:
adata_filtered.obs = adata_filtered.obs.merge(sample_sex[['sex']], left_on='Source Name', right_index=True, how='left')
adata_filtered

AnnData object with n_obs × n_vars = 62586 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library con

## Calculate cell cycle score

In [ ]:
cell_cycle_genes = pd.read_csv('regev_lab_cell_cycle_genes.txt', header=None)

In [163]:
s_genes = cell_cycle_genes.iloc[0:43]
g2m_genes = cell_cycle_genes.iloc[43:]

In [164]:
len(s_genes), len(g2m_genes)

(43, 54)

In [165]:
s_genes_in_adata = [gene for gene in s_genes[0] if gene in adata_filtered.var_names]
g2m_genes_in_adata = [gene for gene in g2m_genes[0] if gene in adata_filtered.var_names]
len(s_genes_in_adata), len(g2m_genes_in_adata)

(42, 52)

In [166]:
adata_log = ad.AnnData(X = adata_filtered.X,  var = adata_filtered.var, obs = adata_filtered.obs)
sc.pp.normalize_total(adata_log, target_sum = 1e6, exclude_highly_expressed = True)
sc.pp.log1p(adata_log)

In [167]:
sc.tl.score_genes_cell_cycle(adata_log, s_genes = s_genes_in_adata, g2m_genes = g2m_genes_in_adata)

In [168]:
adata_filtered.obs['S_score'] = adata_log.obs['S_score']
adata_filtered.obs['G2M_score'] = adata_log.obs['G2M_score']
adata_filtered.obs['Cell_cycle_phase'] = adata_log.obs['phase']

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata_filtered.obs, x='Cell_cycle_phase')
plt.title('Number of Cells per Cell Cycle Phase')
plt.xlabel('Cell Cycle Phase')
plt.ylabel('Number of Cells')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_cell_cycle_phase_{timestamp}_.png")
plt.show()

### Export anndata object

In [170]:
adata_filtered.uns['processing_history']

{'step': 'create raw anndata after mapping, no filtering',
 'timestamp': '05112024_113505'}

In [171]:
current_history = adata_filtered.uns['processing_history']
new_history = [
     json.dumps(current_history),
     json.dumps({
          'timestamp': timestamp,
          'step': 'doublets info; filtered cells with less than 100 genes; filtered cells: pct_mito <50, n_genes>50, n_counts>500, percent_hb<2, total_counts<9000, n_genes_by_counts<5000; added sex covariates and cell cycle phase',
          })
        ]
adata_filtered.uns['processing_history'] = new_history

In [173]:
adata_filtered.uns['processing_history']

['{"step": "create raw anndata after mapping, no filtering", "timestamp": "05112024_113505"}',
 '{"timestamp": "09122024_154541", "step": "doublets info; filtered cells with less than 100 genes; filtered cells: pct_mito <50, n_genes>50, n_counts>500, percent_hb<2, total_counts<9000, n_genes_by_counts<5000; added sex covariates and cell cycle phase"}']

In [175]:
adata_filtered.obs['predicted_doublets'] = adata_filtered.obs['predicted_doublets'].astype(str)

In [177]:
adata_filtered.uns['scautoqc_ranges']['low'] = json.dumps(adata_filtered.uns['scautoqc_ranges']['low'].to_dict())
adata_filtered.uns['scautoqc_ranges']['high'] = json.dumps(adata_filtered.uns['scautoqc_ranges']['high'].to_dict())

In [ ]:
adata_filtered.write_h5ad(f"Processed_data/Elementaite_2021/{project}_{species}_{atribute}_QC_filtered_{name}_{timestamp}_{counts}.h5ad")